In [464]:
import ROOT
import numpy as np
import pandas as pd
 
ROOT.gStyle.SetOptFit(1111)    # aggiunge il label con chi2/ndf e probabilita e parametri se metti 0000 toglie i dati
ROOT.gStyle.SetFitFormat(
    "4.2e"
)  # con 2 cifre decimali


#fit 
def Malus(x, par):
    t = x[0]
    A = par[0]
    B = par[1]
    C = par[2]
    return A*np.cos(t - C)**2 + B

def Malus_fondo(x, par):
    t = x[0]
    A = par[0]
    B = par[1]
    C = par[2]
    D = par[3]
    return (A*np.cos(t - C)**2 + B)*(1 + D*t) 

def modellopy(t, A, B, C):         
    return A*np.cos(t - C)**2 + B

def modellopy_fondo(t, A, B, C, D):         
    return (A*np.cos(t - C)**2 + B)* (1+ D*t )

def Malus_deriv(x, par):
    t = x
    dt = 1e-6
    return ((modellopy(t+dt/2, par[0], par[1], par[2])-modellopy(t-dt/2, par[0], par[1], par[2]))/dt)

def Malus_deriv_fondo(x, par):
    t = x
    dt = 1e-6
    return ((modellopy_fondo(t+dt/2, par[0], par[1], par[2], par[3])-modellopy_fondo(t-dt/2, par[0], par[1], par[2], par[3]))/dt)   


xlsx ="Esperienza4.xlsx"
df = pd.read_excel(xlsx, sheet_name= "Dati")
df = df.dropna(subset=["θ [rad]", "V", "sigma_θ", "sigma_V"])
df = df.to_numpy(dtype = 'float64')

x = df[:, 1]     
y = df[:, 2]     

err_x = df[:, 3]     
err_y = df[:, 4]   

# eliminazione primi dati
index = [i for i in range(13)]

x = np.delete(x, index)
y = np.delete(y, index)
err_x = np.delete(err_x, index)
err_y = np.delete(err_y, index)



#minimizzazzione con ROOT
MalusFit = ROOT.TF1('model', Malus, min(x), max(x), 3)
MalusFit.SetParameters(y.max() - y.min(), y.min(), x[np.argmax(y)])     # A è Vmax-Vmin, B è Vmin, C è posizione del massimo
MalusFit.SetParNames("A", "B", "C")

gr1 = ROOT.TGraphErrors(len(x), x, y, err_x, err_y)
gr1.SetTitle("Dati sperimentali e fit; #theta [rad]; V [V]")
gr1.SetMarkerStyle(20)
gr1.SetMarkerSize(0.7)                    
gr1.SetMarkerColor(ROOT.kAzure + 2)
gr1.SetLineColor(ROOT.kAzure + 3)        
gr1.SetLineWidth(1)


fit_res = gr1.Fit(MalusFit, "S") # SR significa che voglio fare il fit senza disegnare la funzione e senza stampare a video i risultati del fit 

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

par = MalusFit.GetParameters()

errors = MalusFit.GetParErrors()

chi2 = fit_res.Chi2()
ndf = fit_res.Ndf()

err_yeff = np.sqrt(err_y**2 + ((Malus_deriv(x, par)) * err_x)**2) #errore effettivo

print("Parametri stimati:")
print(f"A = {par[0]} ± {errors[0]}")
print(f"B = {par[1]} ± {errors[1]}")
print(f"C = {par[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}, ndf = {ndf}")

c1 = ROOT.TCanvas("c1", "Fit con ROOT", 800, 600)
c1.SetGrid()

gr1.Draw("AP")


leg1 = ROOT.TLegend(0.14, 0.14, 0.42, 0.23)
leg1.AddEntry(gr1, "Dati", "pe")
leg1.AddEntry(MalusFit, "Fit", "l")
leg1.Draw()

c1.Draw()
c1.SaveAs("Immagini/malus_fit.png")


Fit status: 0
Parametri stimati:
A = 1.9706007575566888 ± 0.002968036824497032
B = 2.788484585684793 ± 0.0019349688161533626
C = 3.1204199878288366 ± 0.001036638037536268
Chi2 = 706.3531377896315, ndf = 184
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      706.353
NDf                       =          184
Edm                       =  1.02571e-06
NCalls                    =           66
A                         =       1.9706   +/-   0.00296804  
B                         =      2.78848   +/-   0.00193497  
C                         =      3.12042   +/-   0.00103664  


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: png file Immagini/malus_fit.png has been created


In [465]:
#minimizzazzione con ROOT
print(par[0], par[1], par[2])
MalusFit_fondo = ROOT.TF1('model', Malus_fondo, min(x), max(x), 4)
MalusFit_fondo.SetParameters(par[0], par[1], par[2], 0)
MalusFit_fondo.SetParNames("A", "B", "C", "D")

gr1 = ROOT.TGraphErrors(len(x), x, y, err_x, err_y)
gr1.SetTitle("Dati sperimentali e fit; #theta [rad]; V [V]")
gr1.SetMarkerStyle(20)
gr1.SetMarkerSize(0.7)                    
gr1.SetMarkerColor(ROOT.kAzure + 2)
gr1.SetLineColor(ROOT.kAzure + 3)        
gr1.SetLineWidth(1)


fit_res = gr1.Fit(MalusFit_fondo, "SR") # SR significa che voglio fare il fit senza disegnare la funzione e senza stampare a video i risultati del fit 

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

par_fondo = MalusFit_fondo.GetParameters()

errors_fondo = MalusFit_fondo.GetParErrors()

chi2 = fit_res.Chi2()
ndf = fit_res.Ndf()

err_yeff = np.sqrt(err_y**2 + ((Malus_deriv_fondo(x, par_fondo)) * err_x)**2) #errore effettivo

print("Parametri stimati:")
print(f"A = {par_fondo[0]} ± {errors_fondo[0]}")
print(f"B = {par_fondo[1]} ± {errors_fondo[1]}")
print(f"C = {par_fondo[2]} ± {errors_fondo[2]}")
print(f"D = {par_fondo[3]} ± {errors_fondo[3]}")
print(f"Chi2 = {chi2}, ndf = {ndf}")

c1 = ROOT.TCanvas("c1", "Fit con ROOT", 800, 600)
c1.SetGrid()

gr1.Draw("AP")


leg1 = ROOT.TLegend(0.14, 0.14, 0.42, 0.23)
leg1.AddEntry(gr1, "Dati", "pe")
leg1.AddEntry(MalusFit_fondo, "Fit", "l")
leg1.Draw()

c1.Draw()
c1.SaveAs("Immagini/malus_fit_fondo.png")
ROOT.gStyle.SetOptFit(0000)    

1.9706007575566888 2.788484585684793 3.1204199878288366
Fit status: 0
Parametri stimati:
A = 1.990777834303078 ± 0.003920252719884154
B = 2.799746145718393 ± 0.002394848269333167
C = 3.1195557527346915 ± 0.001040650497758633
D = -0.0014358069376979848 ± 0.00017819425915821933
Chi2 = 641.8391177403386, ndf = 183
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      641.839
NDf                       =          183
Edm                       =  1.73332e-08
NCalls                    =           99
A                         =      1.99078   +/-   0.00392025  
B                         =      2.79975   +/-   0.00239485  
C                         =      3.11956   +/-   0.00104065  
D                         =  -0.00143581   +/-   0.000178194 


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: png file Immagini/malus_fit_fondo.png has been created


In [466]:
if np.abs(par[2]) < 3*errors[2]:
        print("C è compatibile con 0")
else :
        print("C NON è compatibile con 0")

print("---")

if np.abs(par[1]) < 3*errors[1]:
    print("B è compatibile con 0")
else :
    print("B NON è compatibile con 0")
 
 


C NON è compatibile con 0
---
B NON è compatibile con 0


In [467]:

y_attesi = np.array([MalusFit.Eval(x_val) for x_val in x])
residuals_norm = (y - y_attesi) / err_yeff


##NUMERO DI BIN DA USARE (FORMULA BRESSAN)
dev_std = np.std(residuals_norm)
delta_bin = 3.5 * dev_std / (len(residuals_norm))**(1/3)
n_bin= (np.max(residuals_norm) - np.min(residuals_norm))/delta_bin
n_bin = int(np.round(n_bin))
print("numero di bin senza code", n_bin)

##NUMERO DI BIN DA USARE SE PRESENTI CODE (FORMULA BRESSAN)
def IQR(dist):
    return np.percentile(dist, 75) - np.percentile(dist, 25)

delta_bin = 2*IQR(residuals_norm)/ (len(residuals_norm))**(1/3)
n_bin= (np.max(residuals_norm) - np.min(residuals_norm))/delta_bin
n_bin = int(np.round(n_bin))
print("numero di bin con code", n_bin)


gaussian_fit = ROOT.TF1('gaussian', 'gaus', min(residuals_norm), max(residuals_norm), 3)
gaussian_fit.SetParNames("const", "mu", "sigma")
gaussian_fit.SetLineColor(ROOT.kRed + 1)
gaussian_fit.SetLineWidth(3)

hist = ROOT.TH1F("hist", "Istogramma dei dati; x; y", 10, min(residuals_norm), max(residuals_norm))
for val in residuals_norm:
    hist.Fill(val)

gaussian_fit.SetParameters(hist.GetMaximum(), hist.GetMean(), hist.GetRMS())
    
hist.SetLineColor(ROOT.kAzure + 3)
hist.SetLineWidth(2)
hist.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)   
hist.GetXaxis().SetTitle("residuo normalizzato")
hist.GetYaxis().SetTitle("conteggi")
hist.SetTitle("Istogramma residui normalizzati")

r = hist.Fit(gaussian_fit, "S")

fitStatus = r.Status()
print(f"Fit status: {fitStatus}")

parameters = gaussian_fit.GetParameters()
errors = gaussian_fit.GetParErrors()

chi2 = gaussian_fit.GetChisquare()

print("Parametri stimati:")
print(f"const = {parameters[0]} ± {errors[0]}")
print(f"#mu = {parameters[1]} ± {errors[1]}")
print(f"#sigma = {parameters[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")


c2 = ROOT.TCanvas("c2", "Istogramma dei dati", 800, 600)
c2.SetGrid()

hist.Draw()

leg2 = ROOT.TLegend(0, 0.30, 0.15, 0.39)
leg2.AddEntry(hist, "Residui", "f")
leg2.AddEntry(gaussian_fit, "Fit gaussiano", "l")
leg2.Draw()

ROOT.gStyle.SetOptFit(1111)  
c2.Draw()
c2.SaveAs("Immagini/malus_hist.png")


numero di bin senza code 7
numero di bin con code 8
Fit status: 0
Parametri stimati:
const = 27.46113051291443 ± 2.8081631525603385
#mu = 0.42418810654997247 ± 0.21311184629482374
#sigma = 2.321021898449889 ± 0.23028995161957688
Chi2 = 8.562049062286743
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      8.56205
NDf                       =            7
Edm                       =  3.13761e-07
NCalls                    =           69
const                     =      27.4611   +/-   2.80816     
mu                        =     0.424188   +/-   0.213112    
sigma                     =      2.32102   +/-   0.23029      	 (limited)


Warning in <TROOT::Append>: Replacing existing TH1: hist (Potential memory leak).
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c2
Info in <TCanvas::Print>: png file Immagini/malus_hist.png has been created


In [468]:
if np.abs(parameters[0] - 1) < 3*errors[0]:
    print("const è compatibile con 1")
else :
    print("const NON è compatibile con 1")

print("---")

if np.abs(parameters[1]) < 3*errors[1]:
    print("mu è compatibile con 0")
else :
    print("mu NON è compatibile con 0")

print("---")
if np.abs(parameters[2] - 1) < 3*errors[2]:
    print("sigma è compatibile con 1")
else :
    print("sigma NON è compatibile con 1")

const NON è compatibile con 1
---
mu è compatibile con 0
---
sigma NON è compatibile con 1


In [469]:
def line(x, par):
    x = x[0]
    m = par[0]
    q = par[1]
    return m * x + q

residuals = y - y_attesi

line_fit = ROOT.TF1('line', line, min(x), max(x), 2)
line_fit.SetParameters(1, 0)
line_fit.SetParNames("m", "q")
line_fit.SetLineColor(ROOT.kRed + 1)
line_fit.SetLineWidth(2)

gr2 = ROOT.TGraphErrors(len(x), x, residuals, np.zeros_like(residuals), err_yeff)
gr2.SetTitle("Residui in funzione di #theta; #theta [rad]; V - V_{att} [V]")
gr2.SetMarkerStyle(20)
gr2.SetMarkerSize(0.7)
gr2.SetMarkerColor(ROOT.kAzure + 2)
gr2.SetLineColor(ROOT.kAzure + 3)

fit_res = gr2.Fit(line_fit, "S")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

parameters = line_fit.GetParameters()
errors = line_fit.GetParErrors()

chi2 = fit_res.Chi2()

print("Parametri stimati:")
print(f"m = {parameters[0]} ± {errors[0]}")
print(f"q = {parameters[1]} ± {errors[1]}")
print(f"Chi2 = {chi2}")

c3 = ROOT.TCanvas("c3", "Fit con ROOT", 800, 600)
c3.SetGrid()

gr2.Draw("AP")

leg3 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg3.AddEntry(gr2, "Residui", "pe")
leg3.AddEntry(line_fit, "Fit lineare", "l")
leg3.Draw()

c3.Draw()
c3.SaveAs("Immagini/malus_residui.png")

Fit status: 0
Parametri stimati:
m = -0.004187379654922052 ± 0.0006545224579061049
q = 0.015780443417931167 ± 0.0027595381459557034
Chi2 = 661.4906693171874
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      661.491
NDf                       =          185
Edm                       =  2.15472e-19
NCalls                    =           40
m                         =  -0.00418738   +/-   0.000654522 
q                         =    0.0157804   +/-   0.00275954  


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c3
Info in <TCanvas::Print>: png file Immagini/malus_residui.png has been created


In [470]:
# stima    disfunzionalita polarizzatore
Vmax = MalusFit.GetParameter(0) + MalusFit.GetParameter(1)     # A + B
dV_semi = (residuals.max() - residuals.min())/2

print(f"Vmax = {Vmax:.3f} V")
print(f"DeltaV (semiampiezza) = {dV_semi:.3f} V   ->  DeltaV/Vmax = {dV_semi/Vmax:.3f}")
print(f"DeltaV (picco-picco)  = {2*dV_semi:.3f} V   ->  DeltaV/Vmax = {2*dV_semi/Vmax:.3f}")


Vmax = 4.759 V
DeltaV (semiampiezza) = 0.073 V   ->  DeltaV/Vmax = 0.015
DeltaV (picco-picco)  = 0.147 V   ->  DeltaV/Vmax = 0.031


## Secondo Fit per correggere scarica batteria

In [471]:
#residui ma per secondo fit

y_attesi_2 = np.array([MalusFit_fondo.Eval(x_val) for x_val in x])
residuals_norm_2 = (y - y_attesi_2) / err_yeff


##NUMERO DI BIN DA USARE (FORMULA BRESSAN)
dev_std = np.std(residuals_norm_2)
delta_bin = 3.5 * dev_std / (len(residuals_norm_2))**(1/3)
n_bin= (np.max(residuals_norm_2) - np.min(residuals_norm_2))/delta_bin
n_bin = int(np.round(n_bin))
print("numero di bin senza code", n_bin)

##NUMERO DI BIN DA USARE SE PRESENTI CODE (FORMULA BRESSAN)
def IQR(dist):
    return np.percentile(dist, 75) - np.percentile(dist, 25)

delta_bin = 2*IQR(residuals_norm_2)/ (len(residuals_norm_2))**(1/3)
n_bin= (np.max(residuals_norm_2) - np.min(residuals_norm_2))/delta_bin
n_bin = int(np.round(n_bin))
print("numero di bin con code", n_bin)


gaussian_fit = ROOT.TF1('gaussian', 'gaus', min(residuals_norm_2), max(residuals_norm_2), 3)
gaussian_fit.SetParNames("const", "mu", "sigma")
gaussian_fit.SetLineColor(ROOT.kRed + 1)
gaussian_fit.SetLineWidth(3)

hist = ROOT.TH1F("hist", "Istogramma dei dati; x; y", 10, min(residuals_norm_2), max(residuals_norm_2))
for val in residuals_norm_2:
    hist.Fill(val)

gaussian_fit.SetParameters(hist.GetMaximum(), hist.GetMean(), hist.GetRMS())
    
hist.SetLineColor(ROOT.kAzure + 3)
hist.SetLineWidth(2)
hist.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)   
hist.GetXaxis().SetTitle("residuo normalizzato")
hist.GetYaxis().SetTitle("conteggi")
hist.SetTitle("Istogramma residui normalizzati secondo fit")

r = hist.Fit(gaussian_fit, "S")

fitStatus = r.Status()
print(f"Fit status: {fitStatus}")

parameters = gaussian_fit.GetParameters()
errors = gaussian_fit.GetParErrors()

chi2 = gaussian_fit.GetChisquare()

print("Parametri stimati:")
print(f"const = {parameters[0]} ± {errors[0]}")
print(f"#mu = {parameters[1]} ± {errors[1]}")
print(f"#sigma = {parameters[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")


c2 = ROOT.TCanvas("c2", "Istogramma dei dati", 800, 600)
c2.SetGrid()

hist.Draw()

leg2 = ROOT.TLegend(0, 0.30, 0.15, 0.39)
leg2.AddEntry(hist, "Residui", "f")
leg2.AddEntry(gaussian_fit, "Fit gaussiano", "l")
leg2.Draw()

ROOT.gStyle.SetOptFit(1111)  
c2.Draw()
c2.SaveAs("Immagini/malus_hist_fondo.png")

#------------------------------------------------------------
#rettA



def line_2(x, par):
    x = x[0]
    m = par[0]
    q = par[1]
    return m * x + q

residuals_2 = y - y_attesi_2

line_fit = ROOT.TF1('line', line_2, min(x), max(x), 2)
line_fit.SetParameters(1, 0)
line_fit.SetParNames("m", "q")
line_fit.SetLineColor(ROOT.kRed + 1)
line_fit.SetLineWidth(2)

gr2 = ROOT.TGraphErrors(len(x), x, residuals_2, np.zeros_like(residuals_2), err_yeff)
gr2.SetTitle("Residui in funzione di #theta; #theta [rad]; V - V_{att} [V]")
gr2.SetMarkerStyle(20)
gr2.SetMarkerSize(0.7)
gr2.SetMarkerColor(ROOT.kAzure + 2)
gr2.SetLineColor(ROOT.kAzure + 3)

fit_res = gr2.Fit(line_fit, "S")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

parameters = line_fit.GetParameters()
errors = line_fit.GetParErrors()

chi2 = fit_res.Chi2()

print("Parametri stimati:")
print(f"m = {parameters[0]} ± {errors[0]}")
print(f"q = {parameters[1]} ± {errors[1]}")
print(f"Chi2 = {chi2}")

c3 = ROOT.TCanvas("c3", "Fit con ROOT", 800, 600)
c3.SetGrid()

gr2.Draw("AP")

leg3 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg3.AddEntry(gr2, "Residui", "pe")
leg3.AddEntry(line_fit, "Fit lineare", "l")
leg3.Draw()

c3.Draw()
c3.SaveAs("Immagini/malus_residui_fondo.png")



numero di bin senza code 8
numero di bin con code 9
Fit status: 0
Parametri stimati:
const = 29.66775076782846 ± 2.9369254778651417
#mu = 0.4310605841348347 ± 0.17054943546069692
#sigma = 1.8864816005089935 ± 0.13280538842369105
Chi2 = 23.62061349472984
Fit status: 0
Parametri stimati:
m = 0.0005338963882689507 ± 0.0006545224776970469
q = -0.0020233776387450497 ± 0.002759538226157543
Chi2 = 641.1737408714665
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      23.6206
NDf                       =            7
Edm                       =  7.99209e-08
NCalls                    =           69
const                     =      29.6678   +/-   2.93693     
mu                        =     0.431061   +/-   0.170549    
sigma                     =      1.88648   +/-   0.132805     	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      641.174
NDf                       =          185
Ed

Warning in <TROOT::Append>: Replacing existing TH1: hist (Potential memory leak).
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c2
Info in <TCanvas::Print>: png file Immagini/malus_hist_fondo.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c3
Info in <TCanvas::Print>: png file Immagini/malus_residui_fondo.png has been created


## Terzo fit per correggere il polarizzatore


In [472]:
# funzione di fit per correggere disomogeneità del polarizzatore
def Malus_filtro(x, par):
    t = x[0]
    A = par[0]
    B = par[1]
    C = par[2]
    D = par[3]
    E = par[4]
    return (A*np.cos(t - C)**2 + B)* (1 + D*t) *(1 + E*np.cos(2*t - C))

def modellopy_filtro(t, A, B, C, D, E):         
    return (A*np.cos(t - C)**2 + B)* (1+ D*t ) *(1 + E*np.cos(2*t - C))

def Malus_deriv_filtro(x, par):
    t = x
    dt = 1e-6
    return ((modellopy_filtro(t+dt/2, par[0], par[1], par[2], par[3], par[4])-modellopy_filtro(t-dt/2, par[0], par[1], par[2], par[3], par[4]))/dt)




#minimizzazzione con ROOT
MalusFit_filtro = ROOT.TF1('model', Malus_filtro, min(x), max(x), 5)
MalusFit_filtro.SetParameters(par_fondo[0], par_fondo[1], par_fondo[2], par_fondo[3],0 )    
MalusFit_filtro.SetParNames("A", "B", "C", "D", "E")

gr1 = ROOT.TGraphErrors(len(x), x, y, err_x, err_y)
gr1.SetTitle("Dati sperimentali e fit; #theta [rad]; V [V]")
gr1.SetMarkerStyle(20)
gr1.SetMarkerSize(0.7)                    
gr1.SetMarkerColor(ROOT.kAzure + 2)
gr1.SetLineColor(ROOT.kAzure + 3)        
gr1.SetLineWidth(1)


fit_res = gr1.Fit(MalusFit_filtro, "S") # SR significa che voglio fare il fit senza disegnare la funzione e senza stampare a video i risultati del fit 

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

par_filtro = MalusFit_filtro.GetParameters()

errors_filtro = MalusFit_filtro.GetParErrors()

chi2 = fit_res.Chi2()
ndf = fit_res.Ndf()

err_yeff = np.sqrt(err_y**2 + ((Malus_deriv_filtro(x, par_filtro)) * err_x)**2) #errore effettivo

print("Parametri stimati:")
print(f"A = {par_filtro[0]} ± {errors_filtro[0]}")
print(f"B = {par_filtro[1]} ± {errors_filtro[1]}")
print(f"C = {par_filtro[2]} ± {errors_filtro[2]}")
print(f"D = {par_filtro[3]} ± {errors_filtro[3]}")
print(f"E = {par_filtro[4]} ± {errors_filtro[4]}")
print(f"F = {par_filtro[5]} ± {errors_filtro[5]}")
print(f"Chi2 = {chi2}, ndf = {ndf}")

c1 = ROOT.TCanvas("c1", "Fit con ROOT", 800, 600)
c1.SetGrid()

gr1.Draw("AP")


leg1 = ROOT.TLegend(0.14, 0.14, 0.42, 0.23)
leg1.AddEntry(gr1, "Dati", "pe")
leg1.AddEntry(MalusFit_filtro, "Fit", "l")
leg1.Draw()

c1.Draw()
c1.SaveAs("Immagini/malus_fit_filtro.png")

Fit status: 0
Parametri stimati:
A = 2.4635400343737337 ± 0.018864215251241646
B = 2.605512700929532 ± 0.007621107687145747
C = 3.121481011390529 ± 0.00093030992037783
D = -0.0008572721056503959 ± 0.0001779096597354787
E = 0.06294795890612852 ± 0.002433691345078165
F = 0.0 ± 0.0
Chi2 = 177.46687910247417, ndf = 182
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      177.467
NDf                       =          182
Edm                       =  2.73083e-07
NCalls                    =          173
A                         =      2.46354   +/-   0.0188642   
B                         =      2.60551   +/-   0.00762111  
C                         =      3.12148   +/-   0.00093031  
D                         = -0.000857272   +/-   0.00017791  
E                         =     0.062948   +/-   0.00243369  


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: png file Immagini/malus_fit_filtro.png has been created


In [473]:
#residui ma per terzo fit

y_attesi_3 = np.array([MalusFit_filtro.Eval(x_val) for x_val in x])
residuals_norm_3 = (y - y_attesi_3) / err_yeff


##NUMERO DI BIN DA USARE (FORMULA BRESSAN)
dev_std = np.std(residuals_norm_3)
delta_bin = 3.5 * dev_std / (len(residuals_norm_3))**(1/3)
n_bin= (np.max(residuals_norm_3) - np.min(residuals_norm_3))/delta_bin
n_bin = int(np.round(n_bin))
print("numero di bin senza code", n_bin)

##NUMERO DI BIN DA USARE SE PRESENTI CODE (FORMULA BRESSAN)
def IQR(dist):
    return np.percentile(dist, 75) - np.percentile(dist, 25)

delta_bin = 2*IQR(residuals_norm_3)/ (len(residuals_norm_3))**(1/3)
n_bin= (np.max(residuals_norm_3) - np.min(residuals_norm_3))/delta_bin
n_bin = int(np.round(n_bin))
print("numero di bin con code", n_bin)


gaussian_fit = ROOT.TF1('gaussian', 'gaus', min(residuals_norm_3), max(residuals_norm_3), 3)
gaussian_fit.SetParNames("const", "mu", "sigma")
gaussian_fit.SetLineColor(ROOT.kRed + 1)
gaussian_fit.SetLineWidth(3)

hist = ROOT.TH1F("hist", "Istogramma dei dati; x; y", 10, min(residuals_norm_3), max(residuals_norm_3))
for val in residuals_norm_3:
    hist.Fill(val)

gaussian_fit.SetParameters(hist.GetMaximum(), hist.GetMean(), hist.GetRMS())
    
hist.SetLineColor(ROOT.kAzure + 3)
hist.SetLineWidth(2)
hist.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)   
hist.GetXaxis().SetTitle("residuo normalizzato")
hist.GetYaxis().SetTitle("conteggi")
hist.SetTitle("Istogramma residui normalizzati secondo fit")

r = hist.Fit(gaussian_fit, "S")

fitStatus = r.Status()
print(f"Fit status: {fitStatus}")

parameters = gaussian_fit.GetParameters()
errors = gaussian_fit.GetParErrors()

chi2 = gaussian_fit.GetChisquare()

print("Parametri stimati:")
print(f"const = {parameters[0]} ± {errors[0]}")
print(f"#mu = {parameters[1]} ± {errors[1]}")
print(f"#sigma = {parameters[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")


c2 = ROOT.TCanvas("c2", "Istogramma dei dati", 800, 600)
c2.SetGrid()

hist.Draw()

leg2 = ROOT.TLegend(0, 0.30, 0.15, 0.39)
leg2.AddEntry(hist, "Residui", "f")
leg2.AddEntry(gaussian_fit, "Fit gaussiano", "l")
leg2.Draw()

ROOT.gStyle.SetOptFit(1111)  
c2.Draw()
c2.SaveAs("Immagini/malus_hist_filtro.png")

#------------------------------------------------------------
#rettA



def line_3(x, par):
    x = x[0]
    m = par[0]
    q = par[1]
    return m * x + q

residuals_3 = y - y_attesi_3

line_fit = ROOT.TF1('line', line_3, min(x), max(x), 2)
line_fit.SetParameters(1, 0)
line_fit.SetParNames("m", "q")
line_fit.SetLineColor(ROOT.kRed + 1)
line_fit.SetLineWidth(2)

gr2 = ROOT.TGraphErrors(len(x), x, residuals_3, np.zeros_like(residuals_3), err_yeff)
gr2.SetTitle("Residui in funzione di #theta; #theta [rad]; V - V_{att} [V]")
gr2.SetMarkerStyle(20)
gr2.SetMarkerSize(0.7)
gr2.SetMarkerColor(ROOT.kAzure + 2)
gr2.SetLineColor(ROOT.kAzure + 3)

fit_res = gr2.Fit(line_fit, "S")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

parameters = line_fit.GetParameters()
errors = line_fit.GetParErrors()

chi2 = fit_res.Chi2()

print("Parametri stimati:")
print(f"m = {parameters[0]} ± {errors[0]}")
print(f"q = {parameters[1]} ± {errors[1]}")
print(f"Chi2 = {chi2}")

c3 = ROOT.TCanvas("c3", "Fit con ROOT", 800, 600)
c3.SetGrid()

gr2.Draw("AP")

leg3 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg3.AddEntry(gr2, "Residui", "pe")
leg3.AddEntry(line_fit, "Fit lineare", "l")
leg3.Draw()

c3.Draw()
c3.SaveAs("Immagini/malus_residui_filtro.png")



numero di bin senza code 9
numero di bin con code 12
Fit status: 0
Parametri stimati:
const = 39.22789249283891 ± 3.7587883289249757
#mu = -0.062028721311836156 ± 0.08051393896096548
#sigma = 1.016441700228603 ± 0.07015915593813982
Chi2 = 4.216640869599654
Fit status: 0
Parametri stimati:
m = 0.00010115814615235807 ± 0.0006495495026943003
q = -0.0003872628727146221 ± 0.00276640687659655
Chi2 = 177.44262473704606
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      4.21664
NDf                       =            6
Edm                       =  1.14763e-07
NCalls                    =           60
const                     =      39.2279   +/-   3.75879     
mu                        =   -0.0620287   +/-   0.0805139   
sigma                     =      1.01644   +/-   0.0701592    	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      177.443
NDf                       =          18

Warning in <TROOT::Append>: Replacing existing TH1: hist (Potential memory leak).
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c2
Info in <TCanvas::Print>: png file Immagini/malus_hist_filtro.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c3
Info in <TCanvas::Print>: png file Immagini/malus_residui_filtro.png has been created


In [474]:
mask = (np.abs(residuals_3) - err_yeff) > 0
residuals_3_masked = residuals_3[mask]
print(f"Numero di residui fuori dall'intervallo di confidenza: {len(residuals_3_masked)}")
print("percentuale di residui fuori dall'intervallo di confidenza: {:.2f}%".format(len(residuals_3_masked)/len(residuals_3)*100))

Numero di residui fuori dall'intervallo di confidenza: 61
percentuale di residui fuori dall'intervallo di confidenza: 32.62%


In [475]:
# stima    disfunzionalita polarizzatore
Vmax = MalusFit_filtro.GetParameter(0) + MalusFit_filtro.GetParameter(1)    
dV_semi = (residuals_3.max() - residuals_3.min())/2

print(f"Vmax = {Vmax:.3f} V")
print(f"DeltaV (semiampiezza) = {dV_semi:.3f} V   ->  DeltaV/Vmax = {dV_semi/Vmax:.3f}")
print(f"DeltaV (picco-picco)  = {2*dV_semi:.3f} V   ->  DeltaV/Vmax = {2*dV_semi/Vmax:.3f}")


Vmax = 5.069 V
DeltaV (semiampiezza) = 0.043 V   ->  DeltaV/Vmax = 0.009
DeltaV (picco-picco)  = 0.087 V   ->  DeltaV/Vmax = 0.017
